In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

#from pathlib import Path

#path importation
#DATA_PATH = r"C:/Users/HP FOLIO\Downloads\btc-historical-data.csv"

DATA_PATH = r"/Users/user/Group_Lima_BTC_Price_prediction/btc_yfinance_2015_2026.csv"

print("The path has been loaded and ready to be read into data-frame.....")

print("The data cleaning has started...")

#1. Read the data into data frame
df = pd.read_csv(
    DATA_PATH,
    na_values=["$", "?", "NA", "N/A", "", "NaN", "nan"],
    keep_default_na=True
)

print("The BTC historical data csv file has been read into data-frame and ready for exploration...")

df.info()

#2. Replace NAN wil UNKNOWN

#Note: Since "Vol Chg" and "MCap Chg" are the only columns where there is missing values
#Note: we will iterate through the missing columns("Vol Chg" and "MCap Chg") with missing values instead of iterating through the whole columns and fill it will UNKNOWN

missing_columns = ["Vol Chg", "MCap Chg"]

df.head(50)

for col in missing_columns:
    if col in df.columns:
        df[col] = df[col].fillna("UNKNOWN")
    else:
        print(f"The column {col} is is not present in the data frame columns")

print("The missing values have been replaced by the string UNKNOWN")

#3. Dropping of Date Columns since we don't need for the model training
df = df.drop("Date", axis=1)

print("The date column has been dropped from the date-frame")

#4. We want to remove the $ from all the columns where the symbol is used

print("Defining columns with $ symbol.....")

dollar_columns = ["Open", "High", "Low", "Avg", "Close", "Volume", "Market Cap"]

for col in dollar_columns:
    if col in df.columns:
        #df[col] = df[col].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
        df[col] = df[col].str.replace({"$": "", ",": ""})
    else:
        print(f"The column {col} cannot be found")

print("$ and , has been removed")

#5. Removal of whitespaces.

for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

print("The whitespaces have been stripped, Feel free to add naked😂")

#6. Removal of last line from the data-frame because it contains incomplete data and we don't want it to corrupt our model during training

df = df.iloc[0:31]

df.info()

#6. Initialization of model training

print("Intialization of model training....")

#First let's convert the type of each column back to float because they have been converted to str during stripping

for col in df.columns:
    df[col] = df[col].astype(float)

#Note: Row 2 be our "y" and make row 3 - 30 "X".  We are doing this because
#1. We want to train our data with a large data
#2. After the model has been trained we want to test the model with row 0. i.e. we will input row 1 into the model to predict row 0 and cross the predicted values with actual values of row 1

y = df['Close']
print(f"Our y is {y}")
print(f"its length is {len(y)}")

X = df.drop('Close', axis=1)
print(f"Our X is {X}")
print(f"its length is {len(y)}")

print("Our X and y has been defined")

#7. Model training

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mse = mean_squared_error(predictions, y_test)
r2 = r2_score(predictions, y_test)

print(f"The Mean Squared Error is {mse} but lower value is better while the R2 is {r2} but closer to 1.00 is better")


The path has been loaded and ready to be read into data-frame.....
The data cleaning has started...
The BTC historical data csv file has been read into data-frame and ready for exploration...
<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Date        32 non-null     str    
 1   Open        32 non-null     str    
 2   High        32 non-null     str    
 3   Low         32 non-null     str    
 4   Avg         32 non-null     str    
 5   Close       32 non-null     str    
 6   Chg         32 non-null     float64
 7   Volume      32 non-null     str    
 8   Vol Chg     31 non-null     float64
 9   Market Cap  32 non-null     str    
 10  MCap Chg    31 non-null     float64
dtypes: float64(3), str(8)
memory usage: 2.9 KB
The missing values have been replaced by the string UNKNOWN
The date column has been dropped from the date-frame
Defining columns with 